In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
PROJECT_DIR = os.path.abspath("..")

INPUT_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "cleaned",
    "ChargingRecords_Cleaned.csv"
)

OUTPUT_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "cleaned",
    "ChargingRecords_Featured.csv"
)

In [3]:
df = pd.read_csv(INPUT_PATH)

df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,CalculatedDuration
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:00,2022-09-15 23:59:00,185,20.36,185.0
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:00,2022-09-14 21:31:00,90,10.19,90.0
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:00,2022-09-14 19:54:00,60,6.78,60.0
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:00,2022-09-30 00:16:00,344,37.65,344.0
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:00,2022-09-26 00:30:00,300,33.81,300.0


In [4]:
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"],
    errors="coerce"
)

df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"],
    errors="coerce"
)

df["StartDay"] = pd.to_datetime(
    df["StartDay"],
    errors="coerce"
)

df["EndDay"] = pd.to_datetime(
    df["EndDay"],
    errors="coerce"
)

In [5]:
# Extract Date Features
df["Year"] = df["StartDatetime"].dt.year

df["Month"] = df["StartDatetime"].dt.month

df["Day"] = df["StartDatetime"].dt.day

df["DayOfWeek"] = df["StartDatetime"].dt.dayofweek

df["WeekOfYear"] = (
    df["StartDatetime"].dt.isocalendar().week.astype("int")
)

df["Quarter"] = df["StartDatetime"].dt.quarter

In [6]:
df["Hour"] = df["StartDatetime"].dt.hour

df["Minute"] = df["StartDatetime"].dt.minute

df["IsWeekend"] = (
    df["DayOfWeek"].isin([5, 6]).astype(int)
)

In [ ]:
def get_time_of_day(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["TimeOfDay"] = df["Hour"].apply(get_time_of_day)

In [ ]:
# Duration Features
df["DurationHours"] = df["Duration"] / 60

df["DurationCategory"] = pd.cut(
    df["Duration"],
    bins=[0, 60, 120, 240, float("inf")],
    labels=[
        "Short",
        "Medium",
        "Long",
        "Very Long"
    ],
    include_lowest=True
)

In [8]:
# Cyclical Time Features
df["HourSin"] = np.sin(
    2 * np.pi * df["Hour"] / 24
)

df["HourCos"] = np.cos(
    2 * np.pi * df["Hour"] / 24
)

df["MonthSin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["MonthCos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

df["DayOfWeekSin"] = np.sin(
    2 * np.pi * df["DayOfWeek"] / 7
)

df["DayOfWeekCos"] = np.cos(
    2 * np.pi * df["DayOfWeek"] / 7
)

In [9]:

demand_threshold = df["Demand"].median()

df["DemandCategory"] = np.where(
    df["Demand"] >= demand_threshold,
    "High Demand",
    "Normal Demand"
)

print("Demand Threshold:", demand_threshold)

print(df["DemandCategory"].value_counts())

Demand Threshold: 14.1
DemandCategory
High Demand      36439
Normal Demand    36387
Name: count, dtype: int64


In [10]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")

print(df.columns.tolist())

print("\nData Types:")

print(df.dtypes)

Dataset Shape: (72826, 32)

Column Names:
['UserID', 'ChargerID', 'ChargerCompany', 'Location', 'ChargerType', 'StartDay', 'StartTime', 'EndDay', 'EndTime', 'StartDatetime', 'EndDatetime', 'Duration', 'Demand', 'CalculatedDuration', 'Year', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear', 'Quarter', 'Hour', 'Minute', 'IsWeekend', 'DurationHours', 'DurationCategory', 'HourSin', 'HourCos', 'MonthSin', 'MonthCos', 'DayOfWeekSin', 'DayOfWeekCos', 'DemandCategory']

Data Types:
UserID                         int64
ChargerID                      int64
ChargerCompany                 int64
Location                      object
ChargerType                    int64
StartDay              datetime64[ns]
StartTime                     object
EndDay                datetime64[ns]
EndTime                       object
StartDatetime         datetime64[ns]
EndDatetime           datetime64[ns]
Duration                       int64
Demand                       float64
CalculatedDuration           float64
Year      

In [11]:
df.isnull().sum()

UserID                0
ChargerID             0
ChargerCompany        0
Location              0
ChargerType           0
StartDay              0
StartTime             0
EndDay                0
EndTime               0
StartDatetime         0
EndDatetime           0
Duration              0
Demand                0
CalculatedDuration    0
Year                  0
Month                 0
Day                   0
DayOfWeek             0
WeekOfYear            0
Quarter               0
Hour                  0
Minute                0
IsWeekend             0
DurationHours         0
DurationCategory      0
HourSin               0
HourCos               0
MonthSin              0
MonthCos              0
DayOfWeekSin          0
DayOfWeekCos          0
DemandCategory        0
dtype: int64

In [12]:
print("Duplicate Records:", df.duplicated().sum())

Duplicate Records: 0


In [13]:
os.makedirs(
    os.path.dirname(OUTPUT_PATH),
    exist_ok=True
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Feature Engineered Dataset Saved Successfully!")

print("File Location:", OUTPUT_PATH)

print("Final Dataset Shape:", df.shape)

Feature Engineered Dataset Saved Successfully!
File Location: c:\Users\yendl\OneDrive\Documents\GUVI Folder\ev-charging_project\data\cleaned\ChargingRecords_Featured.csv
Final Dataset Shape: (72826, 32)
